In [1]:
print("hello")

hello


In [5]:
# Step 1: Import Required Libraries

from pathlib import Path
import numpy as np
import pandas as pd

In [6]:
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [7]:
from sklearn.model_selection import train_test_split
print("sklearn model_selection imported")

sklearn model_selection imported


In [8]:
from sklearn.preprocessing import StandardScaler
print("StandardScaler imported")

StandardScaler imported


In [9]:
plt.style.use("ggplot")

In [10]:
#Step 2 – Create Feature Engineered Folder
feature_folder = Path("feature_engineered")
feature_folder.mkdir(exist_ok=True)
print(feature_folder)

feature_engineered


In [ ]:
#Step 3 – Locate-Processed Files
processed_2019 = Path("processed_2019")
processed_2020 = Path("processed_2020")

files_2019 = sorted(processed_2019.glob("*.parquet"))
files_2020 = sorted(processed_2020.glob("*.parquet"))

print("2019 Files :", len(files_2019))
print("2020 Files :", len(files_2020))

2019 Files : 2920
2020 Files : 1776


In [12]:
#Step 4 - Wind Speed Function
def calculate_wind_speed(df):

    df = df.copy()

    df["wind_speed"] = np.sqrt(
        df["u_wind"]**2 +
        df["v_wind"]**2
    )

    return df

In [13]:
#Step 5 - Wind Direction Function
def calculate_wind_direction(df):

    df = df.copy()

    direction = (
        270 -
        np.degrees(
            np.arctan2(
                df["v_wind"],
                df["u_wind"]
            )
        )
    ) % 360

    df["wind_direction"] = direction

    return df

In [14]:
#Step 6 – Time Feature Engineering
def create_time_features(df):

    df = df.copy()

    df["time"] = pd.to_datetime(
        df["time"],
        format="%Y%m%d%H"
    )

    df["year"] = df["time"].dt.year

    df["month"] = df["time"].dt.month

    df["day"] = df["time"].dt.day

    df["hour"] = df["time"].dt.hour

    return df

In [15]:
#Step 7 - Season feature engineering
def create_season(df):

    df = df.copy()

    season_map = {
        12: "Winter",
        1: "Winter",
        2: "Winter",
        3: "Spring",
        4: "Spring",
        5: "Spring",
        6: "Summer",
        7: "Summer",
        8: "Summer",
        9: "Autumn",
        10: "Autumn",
        11: "Autumn"
    }

    df["season"] = df["month"].map(season_map)

    return df

In [16]:
#Step 8 – Complete Feature Engineering Function
def engineer_features(df):

    df = calculate_wind_speed(df)

    df = calculate_wind_direction(df)

    df = create_time_features(df)

    df = create_season(df)

    return df

In [17]:
#Step - 9 Test on One File
sample = pd.read_parquet(
    files_2020[0]
)

sample = engineer_features(sample)

sample.head()

,time,pressure_hpa,latitude,longitude,geopotential_height,temperature,relative_humidity,u_wind,v_wind,wind_speed,wind_direction,year,month,day,hour,season
0,2020-01-01,50.0,-15.0,33.96,20731.246094,208.090027,3.403575,-5.746175,-0.919383,5.819260,80.909760,2020,1,1,0,Winter
1,2020-01-01,50.0,-15.0,34.08,20819.746094,209.363464,2.856700,-5.824300,-0.216258,5.828313,87.873535,2020,1,1,0,Winter
2,2020-01-01,50.0,-15.0,34.20,20772.246094,208.793152,3.122325,-5.386800,-0.935008,5.467344,80.153046,2020,1,1,0,Winter
3,2020-01-01,50.0,-15.0,34.32,20539.246094,205.754089,4.716075,-3.996175,-2.966258,4.976756,53.414429,2020,1,1,0,Winter
4,2020-01-01,50.0,-15.0,34.44,20399.246094,203.941589,6.075450,-2.168050,-4.638133,5.119836,25.053284,2020,1,1,0,Winter


In [18]:
#Step 10 - check columns
sample.columns

Index(['time', 'pressure_hpa', 'latitude', 'longitude', 'geopotential_height',
       'temperature', 'relative_humidity', 'u_wind', 'v_wind', 'wind_speed',
       'wind_direction', 'year', 'month', 'day', 'hour', 'season'],
      dtype='str')

In [19]:
#Step 11 - Save one file
sample.to_parquet(
    feature_folder / "sample.parquet",
    index=False
)

In [20]:
#Step 12 – Process Entire Dataset
all_files = files_2019 + files_2020

failed = []

for i, file in enumerate(all_files):

    print(f"[{i+1}/{len(all_files)}] {file.name}")

    try:

        df = pd.read_parquet(file)

        df = engineer_features(df)

        output = feature_folder / file.name

        df.to_parquet(
            output,
            index=False
        )

    except Exception as e:

        failed.append((file.name, str(e)))

print("\nCompleted")

print("Failed :", len(failed))

[1/4696] 2019010100.parquet
[2/4696] 2019010103.parquet
[3/4696] 2019010106.parquet
[4/4696] 2019010109.parquet
[5/4696] 2019010112.parquet
[6/4696] 2019010115.parquet
[7/4696] 2019010118.parquet
[8/4696] 2019010121.parquet
[9/4696] 2019010200.parquet
[10/4696] 2019010203.parquet
[11/4696] 2019010206.parquet
[12/4696] 2019010209.parquet
[13/4696] 2019010212.parquet
[14/4696] 2019010215.parquet
[15/4696] 2019010218.parquet
[16/4696] 2019010221.parquet
[17/4696] 2019010300.parquet
[18/4696] 2019010303.parquet
[19/4696] 2019010306.parquet
[20/4696] 2019010309.parquet
[21/4696] 2019010312.parquet
[22/4696] 2019010315.parquet
[23/4696] 2019010318.parquet
[24/4696] 2019010321.parquet
[25/4696] 2019010400.parquet
[26/4696] 2019010403.parquet
[27/4696] 2019010406.parquet
[28/4696] 2019010409.parquet
[29/4696] 2019010412.parquet
[30/4696] 2019010415.parquet
[31/4696] 2019010418.parquet
[32/4696] 2019010421.parquet
[33/4696] 2019010500.parquet
[34/4696] 2019010503.parquet
[35/4696] 2019010506.pa

In [21]:
#verifying the files
len(list(feature_folder.glob("*.parquet")))

4697

In [24]:
# Inspect one feature-engineered file

sample = pd.read_parquet(next(feature_folder.glob("*.parquet")))

print("Shape:", sample.shape)

sample.head()

Shape: (1833005, 16)


,time,pressure_hpa,latitude,longitude,geopotential_height,temperature,relative_humidity,u_wind,v_wind,wind_speed,wind_direction,year,month,day,hour,season
0,2019-01-01,50.0,-15.0,33.96,20456.568359,202.932434,4.554688,-17.626209,0.869842,17.647659,92.825226,2019,1,1,0,Winter
1,2019-01-01,50.0,-15.0,34.08,20534.068359,203.994934,3.902344,-17.485584,0.885467,17.507990,92.898987,2019,1,1,0,Winter
2,2019-01-01,50.0,-15.0,34.20,20623.568359,205.182434,3.277344,-17.516834,1.033904,17.547319,93.377884,2019,1,1,0,Winter
3,2019-01-01,50.0,-15.0,34.32,20716.568359,206.338684,2.769531,-17.501209,1.174529,17.540577,93.839447,2019,1,1,0,Winter
4,2019-01-01,50.0,-15.0,34.44,20715.068359,206.190247,2.855469,-16.782459,1.315154,16.833912,94.480820,2019,1,1,0,Winter


In [25]:
sample.dtypes

time                   datetime64[ns]
pressure_hpa                  float64
latitude                      float64
longitude                     float64
geopotential_height           float32
temperature                   float32
relative_humidity             float32
u_wind                        float32
v_wind                        float32
wind_speed                    float32
wind_direction                float32
year                            int32
month                           int32
day                             int32
hour                            int32
season                            str
dtype: object

In [26]:
sample.isnull().sum()

time                   0
pressure_hpa           0
latitude               0
longitude              0
geopotential_height    0
temperature            0
relative_humidity      0
u_wind                 0
v_wind                 0
wind_speed             0
wind_direction         0
year                   0
month                  0
day                    0
hour                   0
season                 0
dtype: int64

In [27]:
sample["wind_speed"].describe()

count    1.833005e+06
mean     1.879911e+01
std      1.384493e+01
min      1.687050e-03
25%      8.369844e+00
50%      1.487819e+01
75%      2.550017e+01
max      8.798804e+01
Name: wind_speed, dtype: float64

In [28]:
sample["wind_direction"].describe()

count    1.833005e+06
mean     2.004521e+02
std      9.225747e+01
min      0.000000e+00
25%      9.431104e+01
50%      2.532063e+02
75%      2.713119e+02
max      3.599220e+02
Name: wind_direction, dtype: float64